In [1]:
# ============================================================
# DYNAMIC ML PIPELINE - MVP
#
# בכל הרצה:
# 1. קוראים את פיצ'רי ה-ML
# 2. יוצרים label מתוצאות התרגול
# 3. מוסיפים רשומות חדשות ל-training table
# 4. מאמנים מודל חדש על כל המידע שהצטבר
# 5. שומרים גרסת מודל חדשה
# 6. יוצרים ושומרים תחזיות
# ============================================================

from datetime import datetime

from pyspark.sql.functions import (
    col,
    avg,
    when,
    lit,
    current_timestamp
)

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.functions import vector_to_array


print("=" * 90)
print("DYNAMIC LEARNING DIFFICULTY MODEL - MVP")
print("=" * 90)


# ------------------------------------------------------------
# 1. Model version
# ------------------------------------------------------------

model_version = datetime.now().strftime(
    "learning_difficulty_model_%Y%m%d_%H%M%S"
)

model_path = (
    "/home/iceberg/notebooks/notebooks/models/"
    + model_version
)

print("Model version:", model_version)
print("Model path:", model_path)


# ------------------------------------------------------------
# 2. Read Gold sources
# ------------------------------------------------------------

ml_features_df = spark.table(
    "demo.gold.ml_learning_difficulty_features"
)

practice_attempts_df = spark.table(
    "demo.gold.fact_practice_attempt"
)

print("\nGold ML feature rows:", ml_features_df.count())
print("Practice attempt rows:", practice_attempts_df.count())


# ------------------------------------------------------------
# 3. Create the real outcome / label
#
# struggle_label = 1 when average session score <= 0.5
# struggle_label = 0 otherwise
# ------------------------------------------------------------

session_labels_df = (
    practice_attempts_df
    .groupBy(
        "user_key",
        "topic_key",
        "session_id"
    )
    .agg(
        avg("score")
        .cast("float")
        .alias("session_avg_score")
    )
    .withColumn(
        "struggle_label",
        when(
            col("session_avg_score") <= 0.5,
            lit(1.0)
        )
        .otherwise(lit(0.0))
        .cast("double")
    )
)

print("\nGenerated session labels:")

session_labels_df.orderBy(
    "user_key",
    "topic_key",
    "session_id"
).show(truncate=False)


# ------------------------------------------------------------
# 4. Join point-in-time features with later outcomes
# ------------------------------------------------------------

current_training_rows_df = (
    ml_features_df.alias("f")
    .join(
        session_labels_df.alias("l"),
        [
            "user_key",
            "topic_key",
            "session_id"
        ],
        "inner"
    )
    .select(
        col("user_key").cast("int"),
        col("topic_key").cast("int"),
        col("session_id"),

        col("avg_score_last_7_days").cast("float"),
        col("failure_rate_last_7_days").cast("float"),
        col("hints_used_last_7_days").cast("int"),
        col("avg_attempt_duration").cast("float"),

        col("confidence_before_avg").cast("float"),
        col("confidence_after_avg").cast("float"),
        col("still_confused_rate").cast("float"),
        col("illusion_gap_score").cast("float"),

        col("repeated_mistake_count").cast("int"),

        col("extraction_confidence_avg").cast("float"),
        col("reliability_score_avg").cast("float"),

        col("overall_motivation_avg").cast("float"),
        col("overall_stress_avg").cast("float"),

        col(
            "topic_self_reported_understanding_avg"
        ).cast("float"),

        col("topic_confidence_avg").cast("float"),

        col("session_avg_score").cast("float"),
        col("struggle_label").cast("double"),

        current_timestamp().alias("training_row_created_at")
    )
)

print("\nCurrent labeled training rows:")

current_training_rows_df.orderBy(
    "user_key",
    "topic_key",
    "session_id"
).show(truncate=False)


# ------------------------------------------------------------
# 5. Create accumulated training table
#
# The table keeps labels from previous pipeline runs.
# New learner sessions will be merged into it.
# ------------------------------------------------------------

spark.sql("""
CREATE TABLE IF NOT EXISTS demo.gold.ml_learning_difficulty_training (
    user_key INT,
    topic_key INT,
    session_id STRING,

    avg_score_last_7_days FLOAT,
    failure_rate_last_7_days FLOAT,
    hints_used_last_7_days INT,
    avg_attempt_duration FLOAT,

    confidence_before_avg FLOAT,
    confidence_after_avg FLOAT,
    still_confused_rate FLOAT,
    illusion_gap_score FLOAT,

    repeated_mistake_count INT,

    extraction_confidence_avg FLOAT,
    reliability_score_avg FLOAT,

    overall_motivation_avg FLOAT,
    overall_stress_avg FLOAT,

    topic_self_reported_understanding_avg FLOAT,
    topic_confidence_avg FLOAT,

    session_avg_score FLOAT,
    struggle_label DOUBLE,

    training_row_created_at TIMESTAMP
)
USING iceberg
""")


# ------------------------------------------------------------
# 6. MERGE new labeled sessions into accumulated training data
#
# Existing session = update
# New session      = insert
# ------------------------------------------------------------

current_training_rows_df.createOrReplaceTempView(
    "current_ml_training_rows"
)

spark.sql("""
MERGE INTO demo.gold.ml_learning_difficulty_training AS target
USING current_ml_training_rows AS source

ON target.user_key = source.user_key
AND target.topic_key = source.topic_key
AND target.session_id = source.session_id

WHEN MATCHED THEN UPDATE SET *

WHEN NOT MATCHED THEN INSERT *
""")


training_history_df = spark.table(
    "demo.gold.ml_learning_difficulty_training"
)

print("\nAccumulated training rows:", training_history_df.count())

training_history_df.orderBy(
    "user_key",
    "topic_key",
    "session_id"
).show(truncate=False)


# ------------------------------------------------------------
# 7. Feature columns used by the model
# ------------------------------------------------------------

feature_columns = [
    "avg_score_last_7_days",
    "failure_rate_last_7_days",
    "hints_used_last_7_days",
    "avg_attempt_duration",

    "confidence_before_avg",
    "confidence_after_avg",
    "still_confused_rate",
    "illusion_gap_score",

    "repeated_mistake_count",

    "extraction_confidence_avg",
    "reliability_score_avg",

    "overall_motivation_avg",
    "overall_stress_avg",

    "topic_self_reported_understanding_avg",
    "topic_confidence_avg"
]


# ------------------------------------------------------------
# 8. Fill missing historical information for the MVP model
#
# NULL still remains meaningful in Gold.
# Filling with zero happens only in the ML training copy.
# ------------------------------------------------------------

model_input_df = training_history_df.fillna(
    0.0,
    subset=feature_columns
)


# ------------------------------------------------------------
# 9. Build the feature vector
# ------------------------------------------------------------

assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features",
    handleInvalid="keep"
)

assembled_training_df = assembler.transform(
    model_input_df
)


# ------------------------------------------------------------
# 10. Verify that labels contain enough classes
# ------------------------------------------------------------

label_count = (
    assembled_training_df
    .select("struggle_label")
    .distinct()
    .count()
)

print("\nNumber of label classes:", label_count)

if label_count < 2:
    raise ValueError(
        "The model requires at least two label classes. "
        "Add both struggling and non-struggling learner examples."
    )


# ------------------------------------------------------------
# 11. Train a new model version using all accumulated data
#
# Every future run reads more labeled sessions and retrains.
# ------------------------------------------------------------

classifier = DecisionTreeClassifier(
    featuresCol="features",
    labelCol="struggle_label",
    predictionCol="prediction",
    probabilityCol="probability",
    maxDepth=3,
    minInstancesPerNode=1,
    seed=42
)

model = classifier.fit(
    assembled_training_df
)

print("\nModel training completed.")


# ------------------------------------------------------------
# 12. Save the new model version
# ------------------------------------------------------------

model.write().overwrite().save(
    model_path
)

print("Model saved successfully:")
print(model_path)


# ------------------------------------------------------------
# 13. Generate predictions
#
# In the current MVP, predictions are displayed on the
# accumulated labeled dataset to prove the end-to-end flow.
# With more data, this will be separated into train/test
# and future unlabeled prediction datasets.
# ------------------------------------------------------------

prediction_df = (
    model.transform(assembled_training_df)

    .withColumn(
        "struggle_probability",
        vector_to_array(
            col("probability")
        )[1].cast("float")
    )

    .select(
        "user_key",
        "topic_key",
        "session_id",

        col("struggle_label")
        .cast("double"),

        col("prediction")
        .cast("double"),

        col("struggle_probability"),

        lit(model_version)
        .alias("model_version"),

        current_timestamp()
        .alias("prediction_created_at")
    )
)

print("\nModel predictions:")

prediction_df.orderBy(
    "user_key",
    "topic_key",
    "session_id"
).show(truncate=False)


# ------------------------------------------------------------
# 14. Create prediction history table
# ------------------------------------------------------------

spark.sql("""
CREATE TABLE IF NOT EXISTS demo.gold.ml_learning_difficulty_predictions (
    user_key INT,
    topic_key INT,
    session_id STRING,

    struggle_label DOUBLE,
    prediction DOUBLE,
    struggle_probability FLOAT,

    model_version STRING,
    prediction_created_at TIMESTAMP
)
USING iceberg
""")


# ------------------------------------------------------------
# 15. Save predictions from this model version
# ------------------------------------------------------------

prediction_df.writeTo(
    "demo.gold.ml_learning_difficulty_predictions"
).append()


# ------------------------------------------------------------
# 16. MVP training accuracy
#
# This is training accuracy only.
# With three rows it is NOT a valid quality measurement.
# ------------------------------------------------------------

correct_predictions = (
    prediction_df
    .filter(
        col("prediction")
        == col("struggle_label")
    )
    .count()
)

total_predictions = prediction_df.count()

training_accuracy = (
    correct_predictions / total_predictions
    if total_predictions > 0
    else 0.0
)

print("\nTraining accuracy:", training_accuracy)


# ------------------------------------------------------------
# 17. Final dynamic ML checks
# ------------------------------------------------------------

training_total = spark.table(
    "demo.gold.ml_learning_difficulty_training"
).count()

prediction_total = spark.table(
    "demo.gold.ml_learning_difficulty_predictions"
).count()

saved_model_predictions = (
    spark.table(
        "demo.gold.ml_learning_difficulty_predictions"
    )
    .filter(
        col("model_version") == model_version
    )
    .count()
)

print("\n" + "=" * 90)
print("DYNAMIC ML PIPELINE COMPLETED")
print("=" * 90)

print("Accumulated training rows:", training_total)
print("Current model version:", model_version)
print("Predictions created by current model:", saved_model_predictions)
print("Total saved prediction history rows:", prediction_total)
print("Model saved at:", model_path)

print("\nDynamic behavior:")
print(
    "Each future pipeline run will merge new labeled sessions, "
    "retrain on all accumulated data, and save a new model version."
)

print("=" * 90)

DYNAMIC LEARNING DIFFICULTY MODEL - MVP
Model version: learning_difficulty_model_20260729_075956
Model path: /home/iceberg/notebooks/notebooks/models/learning_difficulty_model_20260729_075956

Gold ML feature rows: 3
Practice attempt rows: 5

Generated session labels:


+--------+---------+-----------+-----------------+--------------+
|user_key|topic_key|session_id |session_avg_score|struggle_label|
+--------+---------+-----------+-----------------+--------------+
|1       |5        |session_001|0.5              |1.0           |
|2       |9        |session_002|0.5              |1.0           |
|3       |8        |session_003|1.0              |0.0           |
+--------+---------+-----------+-----------------+--------------+


Current labeled training rows:
+--------+---------+-----------+---------------------+------------------------+----------------------+--------------------+---------------------+--------------------+-------------------+------------------+----------------------+-------------------------+---------------------+----------------------+------------------+-------------------------------------+--------------------+-----------------+--------------+--------------------------+
|user_key|topic_key|session_id |avg_score_last_7_days|failure_rate_

26/07/29 08:00:04 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.



Accumulated training rows: 3
+--------+---------+-----------+---------------------+------------------------+----------------------+--------------------+---------------------+--------------------+-------------------+------------------+----------------------+-------------------------+---------------------+----------------------+------------------+-------------------------------------+--------------------+-----------------+--------------+--------------------------+
|user_key|topic_key|session_id |avg_score_last_7_days|failure_rate_last_7_days|hints_used_last_7_days|avg_attempt_duration|confidence_before_avg|confidence_after_avg|still_confused_rate|illusion_gap_score|repeated_mistake_count|extraction_confidence_avg|reliability_score_avg|overall_motivation_avg|overall_stress_avg|topic_self_reported_understanding_avg|topic_confidence_avg|session_avg_score|struggle_label|training_row_created_at   |
+--------+---------+-----------+---------------------+------------------------+---------------

26/07/29 08:00:06 WARN DecisionTreeMetadata: DecisionTree reducing maxBins from 32 to 3 (= number of training instances)



Model training completed.


Model saved successfully:
/home/iceberg/notebooks/notebooks/models/learning_difficulty_model_20260729_075956

Model predictions:
+--------+---------+-----------+--------------+----------+--------------------+-----------------------------------------+-------------------------+
|user_key|topic_key|session_id |struggle_label|prediction|struggle_probability|model_version                            |prediction_created_at    |
+--------+---------+-----------+--------------+----------+--------------------+-----------------------------------------+-------------------------+
|1       |5        |session_001|1.0           |1.0       |1.0                 |learning_difficulty_model_20260729_075956|2026-07-29 08:00:09.30228|
|2       |9        |session_002|1.0           |1.0       |1.0                 |learning_difficulty_model_20260729_075956|2026-07-29 08:00:09.30228|
|3       |8        |session_003|0.0           |0.0       |0.0                 |learning_difficulty_model_20260729_075956|2026-07-29